In [ ]:
import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate

#Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Using device: cuda


In [4]:
#Dataset
dataset = load_dataset("dair-ai/emotion")

#Label mapping
labels = dataset["train"].features["label"].names  # ['sadness','joy','love','anger','fear','surprise']
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

print(f"Labels: {labels}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Train: 16000 | Val: 2000 | Test: 2000


In [5]:
checkpoints = [
    "bert-base-uncased",
    "distilbert-base-uncased",
    "roberta-base",
]

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    f1_per_class = f1_metric.compute(predictions=predictions, references=labels, average=None)
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        **{f"f1_{id2label[i]}": round(v, 4) for i, v in enumerate(f1_per_class["f1"])},
    }

In [ ]:
#Guardaremos las métricas en un JSON
all_results = {}

#Training loop
for checkpoint in checkpoints:
    model_name = checkpoint.split("/")[-1]
    print(f"\n{'='*60}")
    print(f"  Training: {checkpoint}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    def tokenize_function(example, tok=tokenizer):
        return tok(example["text"], truncation=True)

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=["text"],
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=f"./results/{model_name}",

        # Epochs & batch
        num_train_epochs=10,               
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,

        # Optimizador
        learning_rate=2e-5,
        weight_decay=0.01,                 
        warmup_steps=0.1,                  
        max_grad_norm=1.0,                 

        # Evaluación y guardado de resultados
        eval_strategy="epoch",             
        save_strategy="epoch",            
        load_best_model_at_end=True,       
        metric_for_best_model="f1",      
        greater_is_better=True,

        # Logging
        logging_strategy="epoch",
        report_to="none",                  

        # Misc
        fp16=torch.cuda.is_available(),    
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2,      
                early_stopping_threshold=0.001, 
            )
        ],
    )

    train_result = trainer.train()

    print(f"\nEvaluando {checkpoint} en test...")
    test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
    from sklearn.metrics import confusion_matrix

    all_preds, all_labels_list = [], []
    for batch in torch.utils.data.DataLoader(
        tokenized_dataset["test"], batch_size=32, collate_fn=data_collator
    ):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = trainer.model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        refs = batch["labels"].cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(refs)

    cm = confusion_matrix(all_labels_list, all_preds)
    all_results[model_name]["confusion_matrix"] = cm.tolist()
            
    test_clean = {k.replace("eval_", ""): v for k, v in test_results.items()}

    print(f"  Accuracy : {test_clean['accuracy']:.4f}")
    print(f"  F1       : {test_clean['f1']:.4f}")

    #Guardar el modelo
    save_path = f"./saved_models/{model_name}"
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Modelo guardado en: {save_path}")

    history = trainer.state.log_history

    train_losses, val_losses, val_accuracies, val_f1s = [], [], [], []
    for entry in history:
        if "loss" in entry and "eval_loss" not in entry:
            train_losses.append(round(entry["loss"], 4))
        if "eval_loss" in entry:
            val_losses.append(round(entry["eval_loss"], 4))
            val_accuracies.append(round(entry.get("eval_accuracy", 0), 4))
            val_f1s.append(round(entry.get("eval_f1", 0), 4))

    #Guardar todos los resultados
    all_results[model_name] = {
        "accuracy": round(test_clean["accuracy"], 4),
        "f1_weighted": round(test_clean["f1"], 4),
        "per_class_f1": [
            round(test_clean.get(f"f1_{id2label[i]}", 0), 4)
            for i in range(len(labels))
        ],
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
        "val_f1s": val_f1s,
        "epochs_trained": len(val_losses), 
        "best_val_f1": round(max(val_f1s) if val_f1s else 0, 4),
        "train_runtime_s": round(train_result.metrics.get("train_runtime", 0), 1),
    }

    # Liberar GPU
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


  Training: bert-base-uncased


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Sadness,F1 Joy,F1 Love,F1 Anger,F1 Fear,F1 Surprise
1,1.119143,0.293335,0.910000,0.910557,0.939100,0.931400,0.842400,0.913500,0.851600,0.829800
2,0.200527,0.160190,0.935500,0.934721,0.961700,0.953600,0.862600,0.938600,0.889400,0.850600
3,0.122326,0.148652,0.944500,0.944297,0.968100,0.959900,0.885500,0.951000,0.898800,0.872100
4,0.096144,0.174226,0.938000,0.938271,0.959600,0.957300,0.891500,0.943400,0.884800,0.853900
5,0.071950,0.196688,0.931500,0.931503,0.954200,0.953000,0.881200,0.934300,0.871900,0.847800


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Evaluando roberta-base en test...


  Accuracy : 0.9350
  F1       : 0.9351


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Modelo guardado en: ./saved_models/roberta-base


In [ ]:
# Guardar en JSON
import os

metrics_payload = {
    "labels": labels,
    "id2label": id2label,
    "models": all_results,
}

os.makedirs("./metrics", exist_ok=True)
with open("./metrics/results.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)

print("\nTerminado!")
print(f"Modelos guardados en  ./saved_models/")
print(f"Metricas guardadas en ./metrics/results.json")

print("RESULTADOS FINALES")
for model_name, results in all_results.items():
    print(
        f"{model_name:30s} | "
        f"Acc: {results['accuracy']:.4f} | "
        f"F1: {results['f1_weighted']:.4f} | "
        f"Epochs: {results['epochs_trained']}"
    )


Terminado!
Modelos guardados en  ./saved_models/
Metricas guardadas en ./metrics/results.json

RESULTADOS FINALES
bert-base-uncased              | Acc: 0.9315 | F1: 0.9307 | Epochs: 6
distilbert-base-uncased        | Acc: 0.9250 | F1: 0.9256 | Epochs: 8
roberta-base                   | Acc: 0.9350 | F1: 0.9351 | Epochs: 7


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

shutil.copytree(
    "/content/saved_models",
    "/content/drive/MyDrive/emotion_project/saved_models",
    dirs_exist_ok=True,
)
shutil.copytree(
    "/content/metrics",
    "/content/drive/MyDrive/emotion_project/metrics",
    dirs_exist_ok=True,
)

print("Guardado en Drive -> emotion_project/")

Mounted at /content/drive
Guardado en Drive -> emotion_project/
